In [21]:
""" Read in coparative datasets and sanity check their labels """
import pandas as pd
comparative_datasets = ['cleaned'] # , 'original', 'augmented']
for dataset in comparative_datasets:
    data = pd.read_csv(f'comparative_datasets/{dataset}/combined_dataset.csv')
    print(f"Dataset: {dataset}, Shape: {data.shape}")
    print(data.columns)


Dataset: cleaned, Shape: (146097, 5)
Index(['src', 'tgt', 'abs_src_FKGL_Grade', 'abs_tgt_FKGL_Grade', 'label'], dtype='object')


In [ ]:
import textstat

def relabel_src_tgt_fkgl(df):
    """Create a new column with src_fkgl, 
              a new column with tgt_fkgl, 
          and a new column with tgt_grade using fkgl rounded down to nearest integer."""
    df['src_fkgl'] = df['src'].apply(lambda x: textstat.flesch_kincaid_grade(x))
    df['tgt_fkgl'] = df['tgt'].apply(lambda x: textstat.flesch_kincaid_grade(x))
    df['tgt_grade'] = df['tgt_fkgl'].apply(lambda x: int(x))  # Round down to nearest integer
    #
    return df

def cleanse(df):
    

In [24]:
raw_data = pd.read_csv(f'comparative_datasets/cleaned/combined_dataset.csv')
from collections import Counter
# count all non-latin characters in src and tgt 
for col in ['src', 'tgt']:
    non_latin_counts = Counter()
    for text in raw_data[col]:
        non_latin_chars = [char for char in text if not char.isascii()]
        non_latin_counts.update(non_latin_chars)
    print(f"Non-latin characters in column '{col}': {non_latin_counts}")

Non-latin characters in column 'src': Counter({'Ã': 7488, 'â': 2964, 'é': 2459, 'Å': 888, 'è': 717, 'á': 716, 'ü': 696, 'Ä': 605, 'Â': 566, 'ö': 556, 'í': 458, 'ó': 439, 'É': 337, 'ä': 310, 'ô': 284, 'Î': 283, 'Ø': 277, 'à': 248, 'ã': 210, 'å': 208, 'ç': 199, 'æ': 196, 'Ù': 191, 'Ð': 185, '°': 184, '²': 167, 'Ë': 148, 'ú': 146, 'ß': 119, 'Ï': 107, 'ñ': 98, 'ë': 88, 'ì': 69, 'Ê': 62, 'Ñ': 61, 'ê': 58, 'ø': 45, 'Á': 45, '×': 43, '\xa0': 42, 'Ö': 41, 'Û': 35, 'ï': 30, 'Õ': 30, 'š': 28, 'Ú': 26, 'î': 26, 'ð': 23, 'ò': 22, '±': 18, 'Ó': 18, 'ù': 16, 'Æ': 15, 'û': 14, '·': 14, 'Ü': 14, 'œ': 13, 'Ç': 13, 'È': 11, '•': 10, '³': 9, 'À': 9, 'Þ': 8, 'ý': 7, '\x92': 7, 'Ž': 7, 'Ì': 6, 'õ': 6, 'ž': 6, 'µ': 5, 'Í': 5, '¥': 4, 'þ': 4, 'ÿ': 3, '†': 3, '§': 3, 'Š': 2, 'ʻ': 2, 'ē': 2, '„': 2, '÷': 2, 'Ò': 2, '©': 1, 'º': 1, 'ā': 1, 'Ô': 1, '’': 1, '´': 1})
Non-latin characters in column 'tgt': Counter({'Ã': 6894, 'â': 2360, 'é': 2017, 'Å': 769, 'ü': 614, 'Â': 579, 'á': 532, 'Ä': 525, 'è': 484, 'ô': 473,

In [ ]:
"""For building the pickle files for each grade from original CSV files"""
from sklearn.model_selection import train_test_split
from collections import defaultdict
from datasets import DatasetDict, Dataset
import pickle
import pandas as pd

def split_grade_pkl_from_full_csv(dataset_type: str):
    print(f"Processing dataset type: {dataset_type}")
    all_data = pd.read_csv(f'comparative_datasets/{dataset_type}/combined_dataset.csv')
    print(f"Loaded data shape: {all_data.shape}")

    all_data = cleanse(all_data)

    all_data = relabel_src_tgt_fkgl(all_data)

    grades = all_data.groupby('abs_tgt_FKGL_Grade')

    grade_dict = defaultdict()
    split_dict = defaultdict()
    for grade in grades:
        grade_dict[grade[0]] = grade[1].reset_index(drop=True)

    for grade in grade_dict.keys():
        if grade < 2 or grade > 12:
            print(f"Skipping grade {grade} due to insufficient data.")
            print(grade_dict[grade].shape)
            continue
        train, test_validate = train_test_split(grade_dict[grade], test_size=1024, random_state=42,)
        train_size = (train.shape[0] // 32 ) * 32
        assert train_size % 32 == 0, "Train size must be a multiple of 32"
        train = train.iloc[:train_size]  # ensure train size is a multiple of 32
        print(f"grade {grade} - Train size: {train.shape[0]}, Test/validate size: {test_validate.shape[0]}")
        validate, test = train_test_split(test_validate, test_size=0.5, random_state=42)
        split_dict[grade] = {'train': train.reset_index(drop=True), 'validate': validate.reset_index(drop=True), 'test': test.reset_index(drop=True)}

    # convert dataframes to datasets and save to pickle format
    for grade in split_dict.keys():
        train_data = Dataset.from_pandas(split_dict[grade]['train'])
        validate_data = Dataset.from_pandas(split_dict[grade]['validate'])
        test_data = Dataset.from_pandas(split_dict[grade]['test'])
        dataset_dict = DatasetDict({'train': train_data, 'validate': validate_data, 'test': test_data})
        # print(dataset_dict)
        print(f"Dataset for grade {grade} created with lengths: train={len(train_data)}, validate={len(validate_data)}, test={len(test_data)}")
        with open(f'comparative_datasets/{dataset_type}/grade{grade:02}/{grade}.pkl', 'wb') as f:
            pickle.dump(dataset_dict, f)
    print(f"All datasets for {dataset_type} saved successfully.")

for dataset_type in ['cleaned']: # , 'original',  'augmented']:
    split_grade_pkl_from_full_csv(dataset_type)

Processing dataset type: original
Loaded data shape: (218367, 5)
Skipping grade 0 due to insufficient data.
Skipping grade 1 due to insufficient data.
grade 2 - Train size: 7328, Test/validate size: 1024
grade 3 - Train size: 9632, Test/validate size: 1024
grade 4 - Train size: 19968, Test/validate size: 1024
grade 5 - Train size: 12832, Test/validate size: 1024
grade 6 - Train size: 24544, Test/validate size: 1024
grade 7 - Train size: 17568, Test/validate size: 1024
grade 8 - Train size: 22688, Test/validate size: 1024
grade 9 - Train size: 15040, Test/validate size: 1024
grade 10 - Train size: 17504, Test/validate size: 1024
grade 11 - Train size: 8128, Test/validate size: 1024
grade 12 - Train size: 9952, Test/validate size: 1024
Skipping grade 13 due to insufficient data.
Dataset for grade 2 created with lengths: train=7328, validate=512, test=512
Dataset for grade 3 created with lengths: train=9632, validate=512, test=512
Dataset for grade 4 created with lengths: train=19968, val

In [10]:
"""Import pickle, convert back to dataframe, export as json in alpaca format"""
import os
from datasets import DatasetDict
import pickle
import pandas as pd
from huggingface_hub import login, HfApi, upload_folder
from dotenv import load_dotenv

load_dotenv()
token = os.getenv("HUGGINGFACE_HUB_TOKEN")
login(token=token)

def reimport_and_save_as_json(grade, dataset_type):
    # Load the pickled dataset

    #system_prompt = "Please rewrite the following sentence to make it easily understandable by students in Grade {tgt_ideal_Grade}. Ensure that the rewritten sentence is grammatically correct, fluent, and retains the core message of the original sentence without changing its meaning."
    instruction_prompt = "Rewrite this Input sentence to make it easily understandable by students in Grade {tgt_ideal_Grade}"# while preserving the meaning: Please note, if the initial rewrite does not meet the specified grade level, you are encouraged to modify and regenerate the output until the criteria are satisfactorily met. The final output should only include the last, correct version of the rewritten sentence(s)."
    
    with open(f'comparative_datasets/{dataset_type}/grade{grade:02}/{grade}.pkl', 'rb') as file:
        data = pickle.load(file)
    
    dataset = DatasetDict()
    
    for split_str in ["train", "validate", "test"]:
        split = pd.DataFrame(data[split_str])
        split['input'] = split['src'].apply(lambda x: x.strip())
        split['output'] = split['tgt'].apply(lambda x: x.strip())

        split['instruction'] = split.apply(lambda x: instruction_prompt.format(tgt_ideal_Grade=grade), axis=1)
  
        split['src_grade'] = split['abs_src_FKGL_Grade']
        split['tgt_grade'] = split['abs_tgt_FKGL_Grade']

        split = split[['input', 'output', 'instruction', 'src_grade', 'tgt_grade', 'label']]
        
        dataset[split_str] = Dataset.from_pandas(split)
    
    # Save as Alpaca-formatted JSON
    path = f'comparative_datasets/{dataset_type}/grade{grade:02}/'
    assert os.path.exists(path), f"Path {path} does not exist."

    for split_str in ["train", "validate", "test"]:
        pd.DataFrame(dataset[split_str]).to_json(f'{path}/{split_str}.json', orient='records', indent=2)
    
    # # Upload JSON files directly to hub to maintain JSON format
    # api = HfApi()
    # repo_id = f'williamplacroix/graded_wikilarge'
    
    # # # # Create repo if it doesn't exist
    # # try:
    # #     api.create_repo(repo_id=repo_id, private=True, exist_ok=True, repo_type="dataset")
    # # except Exception as e:
    # #     print(f"Repo creation note: {e}")
    
    # # Upload each JSON file
    # for split_str in ["train", "validate", "test"]:
    #     # try:
    #     #     api.delete_file(
    #     #         path_in_repo=f'{split_str}.json',
    #     #         repo_id=repo_id,
    #     #         repo_type="dataset",  # or "model"
    #     #     )
    #     # except Exception as e:
    #     #     print(f"nothing to delete: {e}")
    #     api.upload_file(
    #         path_or_fileobj=f'{path}/{split_str}.json',
    #         path_in_repo=f'{split_str}.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update {split_str} split",
    #         repo_type="dataset"
    #     )
    
    print(f"Grade {grade} processed and saved to {path}")

from tqdm import tqdm
# test with grade 8
for dataset_type in ['original', 'cleaned', 'augmented']:
    print(f"Processing dataset type: {dataset_type}")
    for grade in tqdm({2,3,4,5,6,7,8,9,10,11,12}):
        reimport_and_save_as_json(grade, dataset_type)
    print(f"All grades for {dataset_type} processed and saved successfully.")



100%|██████████| 11/11 [00:30<00:00,  2.77s/it]


In [3]:
""" Sample from each grade for building baseline validation and test sets """

from tqdm import tqdm
import pandas as pd
# from datasets import load_dataset

def combine_and_split_baseline(dataset_type) -> None:
    print(f"Combining and splitting baseline for dataset type: {dataset_type}")
    val_sets = []
    test_sets = []
    train_sets = []
    for grade in tqdm(range(2, 13)):
        # with open (f'grade_{grade}/val.json') as f:
        #     grade_data = json.load(f)
        # dataset = load_dataset(f"williamplacroix/graded_wikilarge",
        #                         data_dir="cleaned/grade05",
        #                         split="train")

        # train_dataframe = pd.DataFrame(dataset['train'])
        # val_dataframe = pd.DataFrame(dataset['validate'])
        # test_dataframe = pd.DataFrame(dataset['test'])
        #grade_dataframe = pd.DataFrame(dataset)
        train_dataframe = pd.read_json(f'comparative_datasets/{dataset_type}/grade{grade:02}/train.json')
        val_dataframe = pd.read_json(f'comparative_datasets/{dataset_type}/grade{grade:02}/validate.json')
        test_dataframe = pd.read_json(f'comparative_datasets/{dataset_type}/grade{grade:02}/test.json')

        train_dataframe['label'] = grade
        val_dataframe['label'] = grade
        test_dataframe['label'] = grade
        
        train_sets.append(train_dataframe)
        val_sets.append(val_dataframe)
        test_sets.append(test_dataframe)
        # print(f"Grade {grade} - Train: {len(train_dataframe)}, Val: {len(val_dataframe)}, Test: {len(test_dataframe)}")

    combined_train = pd.concat(train_sets, ignore_index=True)
    combined_val = pd.concat(val_sets, ignore_index=True)
    combined_test = pd.concat(test_sets, ignore_index=True)

    # train_val_sample, _ = train_test_split(combined_train, train_size=len(combined_val), random_state=42, stratify=combined_train['label'])
    # train_test_sample, _ = train_test_split(combined_train, train_size=len(combined_test), random_state=42, stratify=combined_train['label'])
    
    

    # shuffle combined train set
    combined_train = combined_train.sample(frac=1, random_state=42).reset_index(drop=True)
    print(combined_train.shape)
    print(combined_train['label'].value_counts())
    print(combined_val.shape)
    print(combined_val['label'].value_counts())
    print(combined_test.shape)
    print(combined_test['label'].value_counts())

    #pd.DataFrame(combined_train).to_json(f'comparative_datasets/{dataset_type}/baseline/train.json', orient='records', indent=2)
    pd.DataFrame(combined_val).to_json(f'comparative_datasets/{dataset_type}/baseline/validate.json', orient='records', indent=2)
    pd.DataFrame(combined_test).to_json(f'comparative_datasets/{dataset_type}/baseline/test.json', orient='records', indent=2)

    print(f"Baseline training set for {dataset_type} saved successfully.")


    # val, _ = train_test_split(combined_val, train_size=512, random_state=42)#, stratify=train_val_sample['label'])
    # test, _ = train_test_split(combined_test, train_size=512, random_state=42)#, stratify=train_test_sample['label'])
    # # print(val['label'].value_counts())
    # # print(test['label'].value_counts())
    # print(val.shape)
    # print(test.shape)
    # # try:
    # #     os.mkdir(f'./baseline')
    # # except:
    # #     pass 
    # pd.DataFrame(val).to_json(f'comparative_datasets/{dataset_type}/baseline/validate.json', orient='records', indent=2)
    # pd.DataFrame(test).to_json(f'comparative_datasets/{dataset_type}/baseline/test.json', orient='records', indent=2)
    
    # print(f"Baseline validation and test sets for {dataset_type} saved successfully.")
    # datasets.Dataset.from_pandas(val).push_to_hub("williamplacroix/wikilarge_baseline_val")
    # datasets.Dataset.from_pandas(test).push_to_hub("williamplacroix/wikilarge_baseline_test")
    # api = HfApi()
    # repo_id = f'williamplacroix/wikilarge_baseline_alpaca'
    # api.upload_file(
    #         path_or_fileobj='./data/wikilarge/cleaned_graded_splits/wikilarge_baseline_alpaca/validate.json',
    #         path_in_repo='validate.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update validate split",
    #         repo_type="dataset"
    #     )
    # api.upload_file(
    #         path_or_fileobj='./data/wikilarge/cleaned_graded_splits/wikilarge_baseline_alpaca/test.json',
    #         path_in_repo='test.json',
    #         repo_id=repo_id,
    #         commit_message=f"Update test split",
    #         repo_type="dataset"
    #     )
    
for dataset_type in ['cleaned']:#, 'original', 'augmented']:
    combine_and_split_baseline(dataset_type)    

Combining and splitting baseline for dataset type: cleaned


100%|██████████| 11/11 [00:01<00:00,  9.80it/s]


(113184, 6)
label
6     17632
8     16064
10    14176
7     12576
4     11264
9     10272
12     8896
5      8416
11     7200
3      3744
2      2944
Name: count, dtype: int64
(5632, 6)
label
2     512
3     512
4     512
5     512
6     512
7     512
8     512
9     512
10    512
11    512
12    512
Name: count, dtype: int64
(5632, 6)
label
2     512
3     512
4     512
5     512
6     512
7     512
8     512
9     512
10    512
11    512
12    512
Name: count, dtype: int64
Baseline training set for cleaned saved successfully.


In [ ]:
""" Upload all JSON files to Hugging Face Hub """

api = HfApi()
repo_id = f'williamplacroix/graded_wikilarge'

upload_folder(
        repo_id=repo_id,
        repo_type="dataset",
        folder_path="comparative_datasets",
        path_in_repo="",                 # keep same structure
        allow_patterns=["**/*.json"],
        commit_message="Initial upload of 3 variations × 11 grades (+baseline) × 3 splits",
    )

Upload 4 LFS files:   0%|          | 0/4 [00:00<?, ?it/s]























































































































































































































































































































































































































































































































train.json: 100%|██████████| 10.9M/10.9M [00:09<00:00, 1.16MB/s]











train.json: 100%|██████████| 11.5M/11.5M [00:09<00:00, 1.19MB/s]








Upload 4 LFS files:  25%|██▌       | 1/4 [00:09<00:29,  9.88s/it]









train.json: 100%|██████████| 12.8M/12.8M [00:10<00:00, 1.26MB/s]



train.json: 100%|██████████| 14.0M/14.0M [00:10<00:00, 1.33MB/s]]
Upload 4 LFS files: 100%|██████████| 4/4 [00:10<00:00,  2.70s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/williamplacroix/graded_wikilarge/commit/c4bdddf2ddfbece58774782d743771415c13f82e', commit_message='Initial upload of 3 variations × 11 grades (+baseline) × 3 splits', commit_description='', oid='c4bdddf2ddfbece58774782d743771415c13f82e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/williamplacroix/graded_wikilarge', endpoint='https://huggingface.co', repo_type='dataset', repo_id='williamplacroix/graded_wikilarge'), pr_revision=None, pr_num=None)

In [4]:
""" Update a single split later: """

from huggingface_hub import upload_file

grade = "baseline"
split = "train"

for dataset_type in ['cleaned', 'original', 'augmented']:
    upload_file(
        path_or_fileobj=f"comparative_datasets/{dataset_type}/{grade}/{split}.json",
        path_in_repo=f"{dataset_type}/{grade}/{split}.json",
        repo_id="williamplacroix/graded_wikilarge",
        repo_type="dataset",
        commit_message=f"Update {dataset_type} {grade} {split}"
    )

train.json: 100%|██████████| 46.7M/46.7M [00:11<00:00, 4.18MB/s]
train.json: 100%|██████████| 66.9M/66.9M [00:15<00:00, 4.32MB/s]
train.json: 100%|██████████| 91.2M/91.2M [00:19<00:00, 4.57MB/s]


In [9]:
import textstat
text = "In India, an used AK-47 costs $3,800 in black market."
print(text)
print(f'{text}\n'*5)
print("FKGL Grade:", textstat.flesch_kincaid_grade(f'{text}\n'*100))

In India, an used AK-47 costs $3,800 in black market.
In India, an used AK-47 costs $3,800 in black market.
In India, an used AK-47 costs $3,800 in black market.
In India, an used AK-47 costs $3,800 in black market.
In India, an used AK-47 costs $3,800 in black market.
In India, an used AK-47 costs $3,800 in black market.

FKGL Grade: 3.650000000000002


In [10]:
import textstat
import pandas as pd
# read in each json file and compute average FKGL score from cleaned datasets for target grade and label
dataset_type = 'cleaned'
for grade in range(2, 13):
    split = 'train'
    df = pd.read_json(f'comparative_datasets/{dataset_type}/grade{grade:02}/{split}.json')
    df['fkgl_scores'] = df['output'].apply(lambda x: round(textstat.flesch_kincaid_grade(x)))
    df['score_diff'] = abs(df['fkgl_scores'] - df['tgt_grade'])
    avg_fkgl = df['fkgl_scores'].mean()
    print(f"Grade {grade}: tgt average FKGL: {avg_fkgl:.2f}, mean dFKGL: {df['score_diff'].mean():.2f}")
    print(f"FKGL distribution:\n{df['fkgl_scores'].value_counts()}")

Grade 2: tgt average FKGL: 2.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
2    2944
Name: count, dtype: int64
Grade 3: tgt average FKGL: 3.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
3    3744
Name: count, dtype: int64
Grade 4: tgt average FKGL: 4.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
4    11264
Name: count, dtype: int64
Grade 5: tgt average FKGL: 5.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
5    8416
Name: count, dtype: int64
Grade 6: tgt average FKGL: 6.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
6    17632
Name: count, dtype: int64
Grade 7: tgt average FKGL: 7.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
7    12576
Name: count, dtype: int64
Grade 8: tgt average FKGL: 8.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
8    16064
Name: count, dtype: int64
Grade 9: tgt average FKGL: 9.00, mean dFKGL: 0.00
FKGL distribution:
fkgl_scores
9    10272
Name: count, dtype: int64
Grade 10: tgt average FKGL: 10.00, mean dFKGL: 0.00
FKGL di

In [ ]:
textstat.flesch_kincaid_grade(text)

In [18]:
# 1) Install deps
#!pip install ftfy Unidecode

# 2) Write the cleaner
import sys, re, argparse, unicodedata as ud
from unidecode import unidecode
import ftfy

NBSP = "\u00A0"
CONTROL_CATS = {"Cc", "Cf", "Cs", "Co", "Cn"}
GARBAGE = dict.fromkeys(map(ord, ["†","‡","•","∙","◦","�"]), None)

def normalize_unicode(s: str) -> str:
    s = ftfy.fix_text(s)
    s = ud.normalize("NFKC", s)
    s = s.replace(NBSP, " ")
    s = re.sub(r"[\u2000-\u200D\u202F\u205F\u2060]", " ", s)
    return s

def standardize_units(s: str) -> str:
    # normalize degree-like expressions and spacing
    s = re.sub(r"\s*(?:deg|°|º)\s*([CFK])\b",
               lambda m: f" °{m.group(1)}", s, flags=re.I)
    s = re.sub(r"(\d)(°[CFK])\b", r"\1 \2", s)
    return s

def strip_controls_and_garbage(s: str) -> str:
    s = "".join(ch for ch in s if (ud.category(ch) not in CONTROL_CATS) or ch in "\t\n")
    s = s.translate(GARBAGE)
    return s

def tidy_parenthetical_scaffolding(s: str) -> str:
    s = re.sub(r"\(\s*[,;:/\-\|]+\s*\)", "", s)
    s = re.sub(r"\s{2,}", " ", s)
    s = re.sub(r"\s*([,;:])\s*", r"\1 ", s)
    s = re.sub(r"\s*([()])\s*", r"\1", s)
    s = re.sub(r"[–—−]", "-", s)
    s = re.sub(r"\s*-\s*", "-", s)
    return s.strip()

def clean_keep_accents(text: str) -> str:
    s = normalize_unicode(text)
    s = standardize_units(s)
    s = strip_controls_and_garbage(s)
    s = tidy_parenthetical_scaffolding(s)
    return s

def clean_ascii(text: str) -> str:
    s = clean_keep_accents(text)
    s = s.replace("°C","<DEGC>").replace("°F","<DEGF>").replace("°K","<DEGK>")
    s = unidecode(s)
    s = re.sub(r"[^\x09\x0A\x0D\x20-\x7E]", "", s)
    s = s.replace("<DEGC>"," degC").replace("<DEGF>"," degF").replace("<DEGK>"," degK")
    return s

import pandas as pd
dirty_data = pd.read_csv(f'comparative_datasets/cleaned/combined_dataset.csv').sample(100)
dirty_data['cleaned_src'] = dirty_data['src'].apply(lambda x: clean_ascii(x))
dirty_data['cleaned_tgt'] = dirty_data['tgt'].apply(lambda x: clean_ascii(x))

for line in dirty_data[['src', 'cleaned_src']].itertuples():
    print(f"Original: {line.src}\nCleaned:  {line.cleaned_src}\n")





Original: A cartoonist is a person who specializes in drawing cartoons.
Cleaned:  A cartoonist is a person who specializes in drawing cartoons.

Original: Other entries included dozens of Great Walls, pandas and dragons.
Cleaned:  Other entries included dozens of Great Walls, pandas and dragons.

Original: Melody and lyrics "From Me to You" comprises five verses and two bridges.
Cleaned:  Melody and lyrics "From Me to You" comprises five verses and two bridges.

Original: The Sorbian languages are classified under the Slavic branch of the Indo-European languages.
Cleaned:  The Sorbian languages are classified under the Slavic branch of the Indo-European languages.

Original: Châtenay is a commune in the Ain department in eastern France.
Cleaned:  Chatenay is a commune in the Ain department in eastern France.

Original: The low fitness of the hybrids would cause selection to favor assortative mating, which would control hybridization.
Cleaned:  The low fitness of the hybrids would cause